# Import libraries

In [ ]:
import pickle
import glob
from pyscenic.rss import regulon_specificity_scores
from pyscenic.plotting import plot_rss
import matplotlib.pyplot as plt
from adjustText import adjust_text
import seaborn as sns
from pyscenic.binarization import binarize
from cytoolz import compose
import operator as op
from random import  sample
from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2
from pyscenic.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies, load_motifs
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell
from typing import Sequence, Type
from pyscenic.genesig import Regulon, GeneSignature, openfile
from re import search
import os
from dfply import *

In [ ]:
from dask.diagnostics import ProgressBar


# from dask.distributed import Client, progress

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Load settings

In [ ]:
if search("ricard", os.uname()[1]):
    exec(open('/Users/ricard/gastrulation_multiome_10x/settings.py').read())
    exec(open('/Users/ricard/gastrulation_multiome_10x/utils.py').read())
    exec(open('/Users/ricard/gastrulation_multiome_10x/Gavin/pyscenic/utils.py').read())
elif search("ebi", os.uname()[1]):
    exec(open('/homes/ricard/gastrulation_multiome_10x/settings.py').read())
    exec(open('/homes/ricard/gastrulation_multiome_10x/utils.py').read())
    exec(open('/homes/ricard/gastrulation_multiome_10x/Gavin/pyscenic/utils.py').read())
elif search("Workstation",os.uname()[1]):
    exec(open('/home/lijingyu/gastrulation/gastrulation_multiome_10x/utils.py').read())
    exec(open('/home/lijingyu/gastrulation/gastrulation_multiome_10x/settings.py').read())
    exec(open('/home/lijingyu/gastrulation_multiome_10x/Gavin/pyscenic/utils.py').read())
else:
    exit("Computer not recognised")

## Define I/O

Input

In [ ]:
io["DATABASE_FOLDER"] = io["basedir"] + "/results/rna/pyscenic/database"
io["DATABASES_GLOB"] = os.path.join(io["DATABASE_FOLDER"], "mm10_*.mc9nr.feather")
io["MOTIF_ANNOTATIONS_FNAME"] = os.path.join(io["DATABASE_FOLDER"],"motifs-v9-nr.mgi-m0.001-o0.0.tbl")
io["mouse_TFs"] = os.path.join(io["DATABASE_FOLDER"],"mm_mgi_tfs.txt")
io

Output

In [ ]:
# io['modules']=io['outdir']+"/module_gastrulation_sub_new.p"
io["outdir"] = os.path.join(io["basedir"],"results/rna/pyscenic")
# io["adjacency_dataframe"] = os.path.join(io["outdir"],"adjacency_table.csv.gz")
io["adjacencies_outfile"] = os.path.join(io["outdir"],"adjacencies.csv.gz")
io["modules_outfile"] = os.path.join(io["outdir"],"modules.pkl")

## Define options 

In [ ]:
# %%capture
# sc.settings.verbosity = 3
# sc.logging.print_versions()
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(8, 7), facecolor='white')

In [ ]:
opts["samples"] = [
	# "E7.5_rep1",
	# "E7.5_rep2",
	# "E8.0_rep1",
	# "E8.0_rep2",
	# "E8.5_rep1",
	"E8.5_rep2"
]

## Load cell metadata

In [ ]:
metadata = (pd.read_table(io["metadata"]) >>
    mask(X.doublet_call==False, X["sample"].isin(opts["samples"]), X["celltype.mapped"].isin(opts["celltypes"]))
)

In [ ]:
metadata.head()

In [ ]:
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_versions()
sc.set_figure_params(dpi=150, fontsize=10, dpi_save=600)

## Load anndata

In [ ]:
adata = load_adata(io["anndata"], normalise = True, cells=metadata["cell"], filter_lowly_expressed_genes=True)

In [ ]:
adata

In [ ]:
# adata = adata[sample(list(adata.obs_names), 6000), :]

In [ ]:
# adata = adata[~(adata.obs['celltype.mapped'].isin(['Parietal_endoderm', 'ExE_endoderm', 'ExE_ectoderm'])), ]

## Pre-processing 

Remove ribosomal and mitochondrial genes

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('mt-') 
adata.var['ribo'] = adata.var_names.str.startswith(("Rps ","Rpl"))
adata = adata[:,(adata.var["mt"]==False) & (adata.var["ribo"]==False)]
adata.shape

In [ ]:
sc.pp.highly_variable_genes(adata,n_top_genes=4000)
adata = adata[:, adata.var['highly_variable']]

In [ ]:

# regress out total counts per cell and the percentage of mitochondrial genes expressed
# sc.pp.regress_out(adata, ['nFeature_RNA', 'mitochondrial_percent_RNA'])

# scale each gene to unit variance, clip values exceeding SD 10.
# sc.pp.scale(adata, max_value=10)

## Dimensionality reduction

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pp.neighbors(adata, n_pcs=25)

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=['stage', 'celltype.mapped'])

## SCENIC steps

### STEP 1: Gene regulatory network inference, and generation of co-expression modules
#### Phase Ia: GRN inference using the GRNBoost2 algorithm

In [ ]:
tf_names = load_tf_names(io["mouse_TFs"])

In [ ]:
expr = pd.DataFrame(adata.X.todense(), index=adata.obs_names, columns=adata.var_names)
# expr = pd.DataFrame(adata.X.todense()[:500,:500], index=adata.obs_names[:500], columns=adata.var_names[:500])


# expr = expr[range(100),range(100)]
expr.shape

In [ ]:
# instantiate a custom Dask distributed Client
# import bokeh
import dask
from distributed import LocalCluster, Client
local_cluster = LocalCluster(n_workers=1, threads_per_worker=1, dashboard_address=8787)
custom_client = Client(local_cluster)

custom_client

# custom_client.shutdown()

In [ ]:
%%time 
# adjacencies = grnboost2(expression_data=expr, tf_names=tf_names, verbose=True)
adjacencies = grnboost2(expression_data=expr, tf_names=tf_names, verbose=True, client_or_address=custom_client)
adjacencies.head()

In [ ]:
# adjacencies.shape
adjacencies

In [ ]:
adjacencies.to_csv(io["adjacencies_outfile"], sep="\t", index=False)

Load precomputed adjacency


In [ ]:
adjacencies = pd.read_table(io["adjacencies_outfile"], delimiter=",", index_col=0)
adjacencies.head()


In [ ]:
adjacencies = add_correlation(adjacencies, expr)
adjacencies.head()

Create modules from a dataframe containing weighted adjacencies between a TF and its target genes.  
def modules_from_adjacencies(adjacencies, ex_mtx thresholds=(0.75, 0.90), top_n_targets=(50,), top_n_regulators=(5,10,50), min_genes=20, absolute_thresholds=False, rho_dichotomize=True, keep_only_activating=True, rho_threshold=RHO_THRESHOLD, rho_mask_dropouts=False)

In [ ]:
# modules = list(modules_from_adjacencies(adjacencies, expr))

# Save
with open(io['modules_outfile'], 'wb') as f:
    pickle.dump(modules, f)

### STEP 2-3: Regulon prediction aka cisTarget from CLI

In [ ]:
io["DATABASE_FOLDER"] = io["basedir"] + "/results/rna/pyscenic/database"
io["DATABASES_GLOB"] = os.path.join(io["DATABASE_FOLDER"], "mm10_*.mc9nr.feather")
io["MOTIF_ANNOTATIONS_FNAME"] = os.path.join(io["DATABASE_FOLDER"],"motifs-v9-nr.mgi-m0.001-o0.0.tbl")
io["mouse_TFs"] = os.path.join(io["DATABASE_FOLDER"],"mm_mgi_tfs.txt")
io

In [ ]:
db_fnames = glob.glob(io["DATABASES_GLOB"])
db_fnames

In [ ]:
dbs = [RankingDatabase(fname=fname, name=os.path.splitext(os.path.basename(fname))[0]) for fname in db_fnames]
dbs

Calculate all regulons for a given sequence of ranking databases and a sequence of co-expression modules.
    The number of regulons derived from the supplied modules is usually much lower. In addition, the targets of the
    retained modules is reduced to only these ones for which a cis-regulatory footprint is present.


prune2df arguments:

    :param rnkdbs: The sequence of databases.
    :param modules: The sequence of modules.
    :param motif_annotations_fname: The name of the file that contains the motif annotations to use.
    :param rank_threshold: The total number of ranked genes to take into account when creating a recovery curve.
    :param auc_threshold: The fraction of the ranked genome to take into account for the calculation of the AUC
    
    :param nes_threshold: The Normalized Enrichment Score (NES) threshold to select enriched features.
    :param motif_similarity_fdr: The maximum False Discovery Rate to find factor annotations for enriched motifs.
    :param orthologuous_identity_threshold: The minimum orthologuous identity to find factor annotations
        for enriched motifs.
    :param weighted_recovery: Use weights of a gene signature when calculating recovery curves?
    :param num_workers: If not using a cluster, the number of workers to use for the calculation
    :param module_chunksize: The size of the chunk to use when using the dask framework.
    :param client_or_address: The client of IP address of the scheduler when working with dask
    :return: A dataframe.

In [ ]:
with ProgressBar():
    df = prune2df(dbs, modules, io["MOTIF_ANNOTATIONS_FNAME"], client_or_address=custom_client)

In [ ]:
df.head()

In [ ]:
io['pruned_motif_outfile'] = os.path.join(io['outdir'], "pruned_motif_gas_sub_new.csv")
df.to_csv(io['pruned_motif_outfile'])

# Reloading the enriched motifs and regulons from file should be done as follows:
# df = load_motifs(io['pruned_motif_outfile'])

Calculate regulons


In [ ]:
regulons = derive_regulons(
    df,
    db_names=('mm10__refseq-r80__10kb_up_and_down_tss.mc9nr',
              'mm10__refseq-r80__500bp_up_and_100bp_down_tss.mc9nr'))

# Save
io['regulons_outfile'] = os.path.join(io['outdir'],'regulons.pkl')
with open(io['regulons_outfile'], 'wb') as f:
    pickle.dump(regulons, f)

### STEP 4: Cellular enrichment (aka AUCell) from CLI

It is important to check that most cells have a substantial fraction of expressed/detected genes in the calculation of the AUC.
The following histogram gives an idea of the distribution and allows selection of an appropriate threshold.

See [the relevant section in the R tutorial](https://bioconductor.org/packages/devel/bioc/vignettes/AUCell/inst/doc/AUCell.html#build-gene-expression-rankings-for-each-cell) for more information.

By using the default setting for `--auc_threshold` of `0.05`, we see that **1192** genes are selected for the rankings based on the plot below.

In [ ]:
# nGenesDetectedPerCell = np.sum(adata.X > 0, axis=1)
# fig, ax = plt.subplots(1, 1, figsize=(8, 5), dpi=150)
# sns.distplot(nGenesDetectedPerCell, norm_hist=False, kde=False, bins='fd')

In [ ]:
auc_mtx = aucell(expr, regulons, num_workers=1, auc_threshold=0.05)
auc_mtx.head()

In [ ]:
io['auc_outfile'] = os.path.join(io['outdir'],"auc.csv.gz")
auc_mtx.to_csv(io['auc_outfile'])

# Integrate scenic results with scanpy

In [ ]:
adata_scenic = add_scenic_metadata(adata, auc_mtx, regulons)

In [ ]:
adata_scenic.obs.columns

# Plot regulons


In [ ]:
features_to_plot = ['Regulon(Creb3l2)', 'Creb3l2', 'Regulon(Etv6)', 'Etv6']
sc.pl.umap(adata_scenic, color=features_to_plot, color_map='magma', ncols=2)

In [ ]:
io['scenic_adata_outfile']= os.path.join(io['outdir'],"adata_scenic.h5ad")
adata.write(io['scenic_adata_outfile'])